In [0]:
# Create the directory structure in DBFS
dbutils.fs.mkdirs("/FileStoreWasim/hr/raw")

In [0]:
# 2. List files to see if they are in the right spot
display(dbutils.fs.ls("/FileStoreWasim/hr/raw"))

In [0]:
display(dbutils.fs.ls("/FileStore/tables/FileStoreWasim/hr/raw"))

In [0]:
# Define paths
source_dir = "/FileStore/tables/FileStore/tables/FileStoreWasim/hr/raw/"
target_dir = "/FileStoreWasim/hr/raw/"

# Create target directory if it doesn't exist
dbutils.fs.mkdirs(target_dir)

# List of files to move to fulfill Task 1 requirements
files = ["employees.csv", "salaries.csv", "departments.csv", "employees_update.csv", "salaries_batch2.csv"]

for f in files:
    dbutils.fs.mv(f"{source_dir}{f}", f"{target_dir}{f}")
    print(f"Moved {f} to {target_dir}")

# Clean up the deep nested upload directory
dbutils.fs.rm("/FileStore/tables/FileStore/", recurse=True)

# Verify the final setup
display(dbutils.fs.ls(target_dir))

## Ingest Raw CSVs into Delta Tables

In [0]:
from pyspark.sql.functions import current_timestamp, col

# Configuration updated for your specific environment
RAW_PATH = "/FileStoreWasim/hr/raw/"
CATALOG = "filestore"
SCHEMA = "bronze"

# Primary tables for the Lakehouse pipeline
tables = ["employees", "salaries", "departments"]

for table in tables:
    # 1. Read raw CSV with schema enforcement (Modules 3, 4)
    df = (spark.read.format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(f"{RAW_PATH}{table}.csv")
          .select("*", "_metadata.file_path")) # UC-compatible metadata
    
    # 2. Add metadata: _ingested_at and _source_file (Task 2)
    df_bronze = (df.withColumn("_ingested_at", current_timestamp())
                   .withColumnRenamed("file_path", "_source_file"))
    
    # 3. Save to Bronze layer Delta tables
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}")
    
    print(f"Success: Table {table} ingested. Schema displayed below:")
    df_bronze.printSchema()

In [0]:
%sql
-- 1. Add CHECK constraint on employment_type (Task 2) 
ALTER TABLE filestore.bronze.employees 
ADD CONSTRAINT type_check 
CHECK (employment_type IN ('full_time', 'part_time', 'contract'));

-- 2. Verify Delta log is active (Task 2) [cite: 1, 33]
DESCRIBE HISTORY filestore.bronze.employees;

-- 3. Corrected Time Travel syntax (Task 2) 
SELECT * FROM filestore.bronze.employees VERSION AS OF 0;

In [0]:
from pyspark.sql.types import StructType

# Get schema without audit columns to avoid ambiguity
base_schema = spark.table("filestore.bronze.employees").schema
raw_schema = StructType([f for f in base_schema if f.name not in ["_source_file", "_ingested_at"]])

# Use Auto Loader for incremental ingestion (Module 4)
(spark.readStream.format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .schema(raw_schema)
  .option("cloudFiles.schemaLocation", "/FileStoreWasim/hr/checkpoints/bronze_updates")
  .load("/FileStoreWasim/hr/raw/employees_update.csv")
  .select("*", col("_metadata.file_path").alias("_source_file"))
  .withColumn("_ingested_at", current_timestamp())
  .writeStream
  .option("checkpointLocation", "/FileStoreWasim/hr/checkpoints/bronze_updates")
  .trigger(availableNow=True)
  .toTable("filestore.bronze.employees_updates"))

## Silver

In [0]:
%sql
-- Create the silver schema in the filestore catalog
CREATE SCHEMA IF NOT EXISTS filestore.silver;

In [0]:
from pyspark.sql.functions import col, floor, months_between, current_date, row_number
from pyspark.sql.window import Window

# 1. Read Bronze Employees and calculate tenure_years
emp_df = (spark.table("filestore.bronze.employees")
    .withColumn("hire_date", col("hire_date").cast("date"))
    .withColumn("tenure_years", floor(months_between(current_date(), col("hire_date")) / 12)))

# 2. Get latest Salary and compute total monthly compensation
# Logic: Rank by date descending, filter for rank 1, then apply bonus formula
window_spec = Window.partitionBy("employee_id").orderBy(col("effective_date").desc())

latest_salary_df = (spark.table("filestore.bronze.salaries")
    .withColumn("rank", row_number().over(window_spec))
    .filter(col("rank") == 1)
    .withColumn("total_monthly_compensation", col("base_salary") + (col("base_salary") * col("bonus_pct"))))

# 3. Join with departments to add name and location
dept_df = spark.table("filestore.bronze.departments")

silver_enriched_df = (emp_df.join(latest_salary_df, "employee_id", "left")
    .join(dept_df, "department_id", "left")
    .select(
        emp_df["*"], 
        latest_salary_df["total_monthly_compensation"],
        dept_df["department_name"],
        dept_df["location"]
    ))

# 4. Write to Silver layer
# Note: This requires the 'filestore.silver' schema to exist
silver_enriched_df.write.format("delta").mode("overwrite").saveAsTable("filestore.silver.employees_enriched")

print("Success: filestore.silver.employees_enriched created and enriched.")

In [0]:
%sql
-- Enable Change Data Feed (CDF) for tracking row-level changes
ALTER TABLE filestore.silver.employees_enriched SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Optimize for faster querying on common filters
OPTIMIZE filestore.silver.employees_enriched ZORDER BY (department_id, hire_date);

In [0]:
from pyspark.sql.functions import col, lit, current_date

# 1. Clean start
spark.sql("DROP TABLE IF EXISTS filestore.silver.employees_scd")

# 2. Create the table using the schema from the main employees table
# This ensures data types (like employee_id) match perfectly
base_df = spark.table("filestore.bronze.employees")

# 3. Create the table and load the INITIAL baseline (Version 1)
(base_df.select(
    "employee_id", 
    "full_name", 
    "job_title", 
    lit(True).alias("is_current"), 
    col("_ingested_at").cast("date").alias("effective_start_date"), # Use ingestion date as start
    lit(None).cast("date").alias("effective_end_date")
).write.format("delta").mode("overwrite").saveAsTable("filestore.silver.employees_scd"))

print("Baseline loaded into SCD table. Now applying updates...")

# 4. Perform the SCD Type 2 MERGE using the update file
# This expires old titles and prepares for new ones
spark.sql("""
MERGE INTO filestore.silver.employees_scd AS target
USING (
    SELECT *, current_date() as merge_date 
    FROM filestore.bronze.employees_updates
) AS source
ON target.employee_id = source.employee_id AND target.is_current = true
WHEN MATCHED AND target.job_title <> source.job_title THEN
  UPDATE SET 
    target.is_current = false, 
    target.effective_end_date = source.merge_date
""")

# 5. Append the updated records as the new 'Current' versions
updates_df = spark.table("filestore.bronze.employees_updates")
new_versions = updates_df.select(
    "employee_id", 
    "full_name", 
    "job_title", 
    lit(True).alias("is_current"), 
    current_date().alias("effective_start_date"),
    lit(None).cast("date").alias("effective_end_date")
)

new_versions.write.format("delta").mode("append").saveAsTable("filestore.silver.employees_scd")

print("SCD Type 2 complete. Table is now populated with history.")

## gold

In [0]:
%sql
-- Create the Gold Schema for reporting
CREATE SCHEMA IF NOT EXISTS filestore.gold;

-- 1. Department Budget View (Task 4)
CREATE OR REPLACE VIEW filestore.gold.dept_monthly_budget AS
SELECT 
    department_name,
    location,
    ROUND(SUM(total_monthly_compensation), 2) as total_budget,
    COUNT(employee_id) as head_count
FROM filestore.silver.employees_enriched
GROUP BY department_name, location;

-- 2. Seniority & Tenure Report (Task 4)
CREATE OR REPLACE VIEW filestore.gold.employee_tenure_report AS
SELECT 
    full_name,
    department_name,
    tenure_years,
    job_title
FROM filestore.silver.employees_enriched
WHERE tenure_years >= 5
ORDER BY tenure_years DESC;

In [0]:
%sql
SELECT * FROM filestore.silver.employees_scd 
ORDER BY employee_id, effective_start_date;

In [0]:
%sql
-- Final business views
CREATE OR REPLACE VIEW filestore.gold.dept_monthly_budget AS
SELECT 
    department_name,
    location,
    ROUND(SUM(total_monthly_compensation), 2) as total_budget,
    COUNT(employee_id) as head_count
FROM filestore.silver.employees_enriched
GROUP BY department_name, location;

CREATE OR REPLACE VIEW filestore.gold.employee_tenure_report AS
SELECT 
    full_name,
    department_name,
    tenure_years,
    job_title
FROM filestore.silver.employees_enriched
WHERE tenure_years >= 5
ORDER BY tenure_years DESC;